In [1]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Union, Any
from sklearn.metrics.pairwise import cosine_distances
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score,
    homogeneity_score, completeness_score, v_measure_score
)
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, HDBSCAN 
import hdbscan

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## Preprocessing

In [2]:
def ensure_directory_exists(directory_path: str):
    if directory_path and not os.path.exists(directory_path):
        os.makedirs(directory_path, exist_ok=True)
        print(f"Создана директория: {directory_path}")

In [3]:
def filter_tokens_by_frequency(embeddings: Dict, min_freq_threshold: int = 3) -> Tuple:
    print(f"ПОДГОТОВКА ТОКЕНОВ (порог частоты СК >= {min_freq_threshold})")
    print(len(embeddings.items()))

    # Собираем все токены
    all_tokens_info = []
    all_embeddings_list = []
    all_sem_classes = []

    for key, data in embeddings.items():
        sent_id, token_idx = key

        sem_class = data.get('sem_class', '').strip()
        if not sem_class or sem_class == '_':
            continue

        all_tokens_info.append({
            'sentence_id': sent_id,
            'token_idx': token_idx,
            'form': data['form'],
            'lemma': data.get('lemma', ''),
            'sem_class': sem_class,
            'sem_role': data.get('sem_role', '').strip(),
            'upos': data.get('upos', ''),
        })

        all_embeddings_list.append(data['embedding'])
        all_sem_classes.append(sem_class)

    X = np.array(all_embeddings_list, dtype=np.float64)
    true_labels = np.array(all_sem_classes)

    class_counts = Counter(true_labels)

    print(f"Всего токенов: {len(all_tokens_info)}")
    print(f"Уникальных семантических классов: {len(class_counts)}")
    print(f"Размерность эмбеддингов: {X.shape}")

    filtered_indices = []
    filtered_classes = []
    filtered_tokens_info = []

    for i, sem_class in enumerate(true_labels):
        if class_counts[sem_class] >= min_freq_threshold:
            filtered_indices.append(i)
            filtered_classes.append(sem_class)
            filtered_tokens_info.append(all_tokens_info[i])

    X_filtered = X[filtered_indices]
    true_labels_filtered = np.array(filtered_classes)

    print(f"После фильтрации:")
    print(f"  Токенов: {len(X_filtered)}")
    print(f"  Уникальных классов: {len(np.unique(true_labels_filtered))}")

    return X_filtered, true_labels_filtered, filtered_tokens_info, filtered_indices, class_counts

In [4]:
def normalize_embeddings(X: np.ndarray, method: str = 'standard') -> np.ndarray:
    if method is None or method == 'none':
        return X
    
    print(f"\nНормализация эмбеддингов: {method}")
    
    if method == 'standard':
        scaler = StandardScaler()
        X_norm = scaler.fit_transform(X)
    elif method == 'minmax':
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        X_norm = scaler.fit_transform(X)
    elif method == 'l2':
        from sklearn.preprocessing import normalize
        X_norm = normalize(X, norm='l2')
    else:
        raise ValueError(f"Unknown normalization method: {method}")
    
    return X_norm

In [6]:
def reduce_dimensions(X: np.ndarray, method: str = 'pca', n_components: int = 50, 
                     random_state: int = 42) -> Tuple[np.ndarray, Dict]:
    info = {'original_shape': X.shape, 'method': method}
    
    if method is None or method == 'none' or X.shape[1] <= n_components:
        info['reduced_shape'] = X.shape
        info['explained_variance'] = 1.0
        return X, info
    
    print(f"\nСнижение размерности: {method} -> {n_components} компонент")
    
    if method == 'pca':
        pca = PCA(n_components=min(n_components, X.shape[0]), random_state=random_state)
        X_reduced = pca.fit_transform(X)
        print(X_reduced.dtype)
        info['explained_variance'] = pca.explained_variance_ratio_.sum()
        info['components'] = pca.components_
        print(f"  Сохранено дисперсии: {info['explained_variance']:.2%}")
    elif method == 'tsne':
        tsne = TSNE(
            n_components=n_components,
            random_state=random_state,
            init='pca',
            learning_rate='auto',
            perplexity=min(30, max(5, X.shape[0] // 100))
        )

        X_reduced = tsne.fit_transform(X)
        info['explained_variance'] = None
        print(f"  t-SNE завершён")    
    
    elif method == 'truncated_svd':
        from sklearn.decomposition import TruncatedSVD
        svd = TruncatedSVD(n_components=min(n_components, X.shape[1]), random_state=random_state)
        X_reduced = svd.fit_transform(X)
        info['explained_variance'] = svd.explained_variance_ratio_.sum()
        print(f"  Сохранено дисперсии: {info['explained_variance']:.2%}")
    
    else:
        raise ValueError(f"Unknown reduction method: {method}")
    
    info['reduced_shape'] = X_reduced.shape
    return X_reduced, info

## Clustering setting

In [7]:
class ClusteringFactory:
    """Фабрика для создания моделей кластеризации"""
    
    @staticmethod
    def create_kmeans_params(n_clusters: int = 10, **kwargs) -> Dict:
        params = {
            'n_clusters': n_clusters,
            'random_state': kwargs.get('random_state', 42),
            'n_init': kwargs.get('n_init', 10),
            'max_iter': kwargs.get('max_iter', 300)
        }
        return params
    
    @staticmethod
    def create_dbscan_params(eps: float = 0.5, min_samples: int = 5, **kwargs) -> Dict:
        params = {
            'eps': eps,
            'min_samples': min_samples,
            'metric': kwargs.get('metric', 'euclidean')
        }
        return params
    
    @staticmethod
    def create_hdbscan_params(min_cluster_size: int = 15, min_samples: Optional[int] = None, 
                             metric: str = 'euclidean', algorithm='kd_tree', **kwargs) -> Dict:
        params = {
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'metric': metric,
            'cluster_selection_epsilon': kwargs.get('cluster_selection_epsilon', 0.0),
            'algorithm': kwargs.get('algorithm', 'kd_tree'),
            'copy': kwargs.get('copy', False),
            'allow_single_cluster': True,
            'prediction_data':True
        }
        return params
    
    @staticmethod
    def create_agglomerative_params(n_clusters: int = 10, linkage: str = 'ward', 
                                   metric: str = 'euclidean', **kwargs) -> Dict:
        params = {
            'n_clusters': n_clusters,
            'linkage': linkage,
            'metric': metric,
            'compute_full_tree': kwargs.get('compute_full_tree', 'auto')
        }
        return params

In [9]:
def perform_clustering(X: np.ndarray, method: str = 'hdbscan', 
                      params: Optional[Dict] = None, 
                      true_labels: Optional[np.ndarray] = None) -> Tuple:
    print("\n" + "=" * 80)
    print(f"КЛАСТЕРИЗАЦИЯ: {method.upper()}")
    print(f"Размер данных: {X.shape}")
    print(X.dtype)
    
    if params:
        print("Параметры:")
        for k, v in params.items():
            print(f"  {k}: {v}")
    try:
        if method == 'kmeans':
            clusterer = KMeans(**params)
            cluster_labels = clusterer.fit_predict(X)
            
        elif method == 'dbscan':
            clusterer = DBSCAN(**params)
            cluster_labels = clusterer.fit_predict(X)
            
        elif method == 'hdbscan':
            # clusterer = HDBSCAN(**params)
            # X = X.astype(np.float64)
            # distance_matrix = cosine_distances(X).astype(np.float64)
            clusterer = hdbscan.HDBSCAN(**params)
            cluster_labels = clusterer.fit_predict(X.astype(np.float64))
            
        elif method == 'agglomerative':
            clusterer = AgglomerativeClustering(**params)
            cluster_labels = clusterer.fit_predict(X)
            
        else:
            raise ValueError(f"Unknown clustering method: {method}")
            
    except Exception as e:
        print(f"Ошибка при кластеризации: {e}")
        return None, None, None
    
    unique_clusters = np.unique(cluster_labels)
    n_clusters_found = len(unique_clusters) - (1 if -1 in unique_clusters else 0)
    n_noise = sum(cluster_labels == -1) if -1 in unique_clusters else 0
    
    print(f"\nРезультаты кластеризации:")
    print(f"  Найдено кластеров: {n_clusters_found}")
    if n_noise > 0:
        print(f"  Шумовых точек: {n_noise} ({n_noise/len(cluster_labels)*100:.1f}%)")
    
    if n_clusters_found > 0 and n_clusters_found < len(X):
        cluster_sizes = [sum(cluster_labels == i) for i in unique_clusters if i != -1]
        print(f"  Размер кластеров: мин={min(cluster_sizes)}, макс={max(cluster_sizes)}, "
              f"ср={np.mean(cluster_sizes):.1f}")
    
    metrics = calculate_clustering_metrics(X, cluster_labels, true_labels)
    if method == 'hdbscan' and hasattr(clusterer, 'cluster_persistence_'):
        metrics['persistence'] = clusterer.cluster_persistence_
    
    return cluster_labels, clusterer, metrics

## Metrics

In [10]:
def calculate_clustering_metrics(X: np.ndarray, cluster_labels: np.ndarray, 
                                 true_labels: Optional[np.ndarray] = None) -> Dict:
    metrics = {}
    
    unique_clusters = np.unique(cluster_labels)
    metrics['n_clusters'] = len(unique_clusters) - (1 if -1 in unique_clusters else 0)
    metrics['n_noise'] = sum(cluster_labels == -1) if -1 in unique_clusters else 0
    metrics['noise_percent'] = metrics['n_noise'] / len(cluster_labels) * 100 if len(cluster_labels) > 0 else 0

    mask = cluster_labels != -1
    if metrics['n_clusters'] > 1 and mask.sum() > 1:
        try:
            metrics['silhouette'] = silhouette_score(X[mask], cluster_labels[mask], metric='euclidean')
            metrics['calinski'] = calinski_harabasz_score(X[mask], cluster_labels[mask])
            metrics['davies'] = davies_bouldin_score(X[mask], cluster_labels[mask])
        except Exception as e:
            print(f"Не удалось вычислить метрики кластеризации: {e}")
    
    if true_labels is not None:
        label_encoder = LabelEncoder()
        try:
            encoded_true = label_encoder.fit_transform(true_labels)
            
            metrics['ari'] = adjusted_rand_score(encoded_true, cluster_labels)
            metrics['nmi'] = normalized_mutual_info_score(encoded_true, cluster_labels)
            metrics['homogeneity'] = homogeneity_score(encoded_true, cluster_labels)
            metrics['completeness'] = completeness_score(encoded_true, cluster_labels)
            metrics['v_measure'] = v_measure_score(encoded_true, cluster_labels)
        except Exception as e:
            print(f"Не удалось вычислить метрики соответствия: {e}")
    
    print("\nМетрики:")
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
    
    return metrics

## Analyze

In [11]:
def analyze_clusters_content(tokens_info: List[Dict], cluster_labels: np.ndarray, 
                            true_labels: np.ndarray, top_n_clusters: int = 15) -> Tuple[pd.DataFrame, Dict]:
    print("\n" + "=" * 80)
    print(f"АНАЛИЗ СОДЕРЖИМОГО КЛАСТЕРОВ (топ-{top_n_clusters})")

    df = pd.DataFrame(tokens_info)
    df['cluster'] = cluster_labels
    df['sem_class'] = true_labels

    noise_df = df[df['cluster'] == -1]
    if len(noise_df) > 0:
        print(f"\nШУМОВЫЕ ТОЧКИ (cluster -1): {len(noise_df)}")
        noise_classes = noise_df['sem_class'].value_counts().head(10)
        print("  Топ семантических классов в шуме:")
        for cls, cnt in noise_classes.items():
            print(f"    {cls}: {cnt}")

    df_clusters = df[df['cluster'] != -1]
    
    if len(df_clusters) == 0:
        print("\nНет кластеров для анализа (все точки в шуме)")
        return df, {}

    cluster_sizes = df_clusters['cluster'].value_counts()
    total_clusters = len(cluster_sizes)

    print(f"\nКЛАСТЕРЫ:")
    print(f"  Всего кластеров: {total_clusters}")
    print(f"  Всего токенов в кластерах: {len(df_clusters)}")
    print(f"  Средний размер кластера: {cluster_sizes.mean():.1f}")

    top_clusters = cluster_sizes.head(top_n_clusters)

    print(f"\nТоп-{top_n_clusters} кластеров по размеру:")

    cluster_details = {}
    for cluster_id, size in top_clusters.items():
        cluster_data = df_clusters[df_clusters['cluster'] == cluster_id]
        top_classes = cluster_data['sem_class'].value_counts().head(3)
        top_forms = cluster_data['form'].value_counts().head(5)
        unique_classes = cluster_data['sem_class'].nunique()
        purity = top_classes.iloc[0] / size if len(top_classes) > 0 else 0

        cluster_details[cluster_id] = {
            'size': size,
            'unique_classes': unique_classes,
            'purity': purity,
            'top_classes': dict(top_classes),
            'top_forms': list(top_forms.index)
        }

        print(f"\nКластер {cluster_id} (размер: {size}):")
        print(f"  Уникальных СК: {unique_classes}")
        print(f"  Чистота: {purity:.1%}")
        print(f"  Топ СК:")
        for cls, count in top_classes.items():
            print(f"    {cls}: {count} ({count/size*100:.1f}%)")
        if not top_forms.empty:
            print(f"  Топ форм: {', '.join(cluster_details[cluster_id]['top_forms'][:5])}")

    purities = [details['purity'] for details in cluster_details.values()]
    if purities:
        print(f"\nСтатистика чистоты (топ-{top_n_clusters}):")
        print(f"  Средняя: {np.mean(purities):.1%}")
        print(f"  Медиана: {np.median(purities):.1%}")

    return df, cluster_details

In [12]:
def plot_metrics_comparison(metrics_dict: Dict, save_path: Optional[str] = None):
    if not metrics_dict:
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    metrics_to_plot = ['ari', 'v_measure', 'homogeneity', 'completeness', 'silhouette', 'n_clusters']
    titles = ['ARI', 'V-measure', 'Homogeneity', 'Completeness', 'Silhouette', 'Number of Clusters']
    
    for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
        ax = axes[i]
        
        values = []
        labels = []
        for exp_name, exp_metrics in metrics_dict.items():
            if metric in exp_metrics and exp_metrics[metric] is not None:
                values.append(exp_metrics[metric])
                labels.append(exp_name)
        
        if values:
            bars = ax.bar(range(len(values)), values)
            ax.set_xticks(range(len(values)))
            ax.set_xticklabels(labels, rotation=45, ha='right')
            ax.set_title(title)
            ax.set_ylabel(title)
            for j, (bar, val) in enumerate(zip(bars, values)):
                if isinstance(val, float):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                           f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(os.path.join(save_path, 'metrics_comparison.png'), dpi=150, bbox_inches='tight')
    
    plt.show()

In [13]:
def save_results(results: Dict, experiment_dir: str, experiment_name: str = 'clustering'):
    print(f"\nСохранение результатов в {experiment_dir}")
    
    def convert_to_serializable(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        elif isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_serializable(item) for item in obj]
        else:
            return obj
    
    results_to_save = {k: v for k, v in results.items() 
                      if k not in ['X_raw', 'X', 'X_tsne', 'clusterer', 'df_clusters']}
    
    with open(os.path.join(experiment_dir, 'results.pkl'), 'wb') as f:
        pickle.dump(results_to_save, f)
    
    if 'df_clusters' in results and results['df_clusters'] is not None:
        results['df_clusters'].to_csv(os.path.join(experiment_dir, 'clusters_data.csv'), index=False)
    
    summary = {
        'experiment_name': experiment_name,
        'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
        'parameters': convert_to_serializable(results.get('params', {})),
        'data_info': {
            'n_tokens': int(len(results.get('X_raw', []))) if 'X_raw' in results else 0,
            'n_tokens_after_reduction': int(len(results.get('X', []))) if 'X' in results else 0,
            'n_classes': int(len(np.unique(results.get('true_labels', [])))) if 'true_labels' in results else 0,
            'original_dim': int(results.get('X_raw', np.array([])).shape[1]) if 'X_raw' in results else 0,
            'reduced_dim': int(results.get('X', np.array([])).shape[1]) if 'X' in results else 0,
        },
        'metrics': convert_to_serializable(results.get('metrics', {}))
    }
    
    if 'reduction_info' in results:
        summary['dimension_reduction'] = convert_to_serializable(results['reduction_info'])
    
    with open(os.path.join(experiment_dir, 'summary.json'), 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    if 'cluster_details' in results and results['cluster_details']:
        cluster_info = []
        for cluster_id, details in results['cluster_details'].items():
            cluster_info.append({
                'cluster_id': int(cluster_id),
                'size': int(details['size']),
                'unique_classes': int(details['unique_classes']),
                'purity': float(details['purity']),
                'top_classes': str(details['top_classes']),
                'top_forms': ', '.join(details['top_forms'][:5])
            })
        
        pd.DataFrame(cluster_info).to_csv(os.path.join(experiment_dir, 'cluster_details.csv'), index=False)
    
    print(f"Результаты сохранены в {experiment_dir}")
    return experiment_dir

## visual

In [14]:
def visualize_confusion_matrix(cluster_labels: np.ndarray, true_labels: np.ndarray,
                              top_n_classes: int = 20, save_path: Optional[str] = None):
    print("\n" + "=" * 80)
    print(f"МАТРИЦА CONFUSION (топ-{top_n_classes} классов)")

    if save_path:
        ensure_directory_exists(save_path)

    class_counts = Counter(true_labels)
    top_classes = [cls for cls, _ in class_counts.most_common(top_n_classes)]

    mask = np.isin(true_labels, top_classes)
    filtered_true = true_labels[mask]
    filtered_clusters = cluster_labels[mask]

    df_conf = pd.DataFrame({
        'sem_class': filtered_true,
        'cluster': filtered_clusters
    })

    pivot = df_conf.pivot_table(
        index='sem_class',
        columns='cluster',
        aggfunc='size',
        fill_value=0
    )

    pivot = pivot.loc[[c for c in top_classes if c in pivot.index]]
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    sns.heatmap(pivot, ax=axes[0], cmap='Blues',
                cbar_kws={'label': 'Количество'})
    axes[0].set_title('Матрица confusion', fontsize=14)
    axes[0].set_xlabel('Кластер')
    axes[0].set_ylabel('Семантический класс')

    sns.heatmap(pivot_norm, ax=axes[1], cmap='Blues',
                cbar_kws={'label': 'Доля'}, vmin=0, vmax=1)
    axes[1].set_title('Матрица confusion (нормализованная)', fontsize=14)
    axes[1].set_xlabel('Кластер')
    axes[1].set_ylabel('Семантический класс')

    plt.suptitle(f'Соответствие кластеров и семантических классов', fontsize=16)
    plt.tight_layout()
    
    if save_path:
        filepath = os.path.join(save_path, 'confusion_matrix.png')
        plt.savefig(filepath, dpi=150, bbox_inches='tight')
        print(f"Матрица confusion сохранена: {filepath}")
    
    plt.show()

    print(f"\nАнализ для топ-{min(10, len(top_classes))} классов:")
    for cls in top_classes[:10]:
        if cls in pivot.index:
            cls_data = df_conf[df_conf['sem_class'] == cls]
            total = len(cls_data)
            if total > 0:
                valid_clusters = cls_data[cls_data['cluster'] != -1]
                if len(valid_clusters) > 0:
                    main_cluster = valid_clusters['cluster'].mode().iloc[0]
                    main_count = (valid_clusters['cluster'] == main_cluster).sum()
                    main_pct = main_count / total * 100
                else:
                    main_cluster = -1
                    main_count = total
                    main_pct = 100
                
                noise_count = (cls_data['cluster'] == -1).sum()
                print(f"\n  {cls}: токенов={total}, осн.кластер={main_cluster} ({main_pct:.1f}%), "
                      f"шум={noise_count} ({noise_count/total*100:.1f}%)")

In [15]:
import matplotlib
import random

def generate_distinct_colors(n_colors: int, seed=42) -> List:
    import colorsys
    
    colors = []
    for i in range(n_colors):
        hue = i / n_colors
        saturation = 0.7
        value = 0.9
        rgb = colorsys.hsv_to_rgb(hue, saturation, value)
        colors.append(rgb)
    return colors

def generate_distinct_colors(n_colors: int, seed=42) -> List:
    import numpy as np
    
    n_per_axis = int(np.ceil(n_colors ** (1/3)))
    
    colors = []
    step = 1.0 / (n_per_axis - 1) if n_per_axis > 1 else 1.0
    
    for r in range(n_per_axis):
        for g in range(n_per_axis):
            for b in range(n_per_axis):
                if len(colors) < n_colors:
                    r_shift = (r * 0.1) % 0.2
                    g_shift = (g * 0.1) % 0.2
                    b_shift = (b * 0.1) % 0.2
                    
                    colors.append((
                        min(1.0, r * step + r_shift),
                        min(1.0, g * step + g_shift),
                        min(1.0, b * step + b_shift)
                    ))
    
    random.seed(42)
    random.shuffle(colors)
    return colors[:n_colors]

In [16]:
def visualize_clusters_2d(X: np.ndarray, cluster_labels: np.ndarray, 
                                          true_labels: np.ndarray, 
                                          save_path: Optional[str] = None) -> np.ndarray:
    print("\n" + "=" * 80)
    print("2D ВИЗУАЛИЗАЦИЯ КЛАСТЕРОВ")

    if save_path:
        ensure_directory_exists(save_path)
    X_reduced = X
    perplexity = 50
    print(f"t-SNE с perplexity={perplexity}...")
    
    tsne = TSNE(n_components=2, 
                             random_state=100, 
                             perplexity=50,
                             max_iter=1000, 
                             learning_rate='auto', 
                             init='random', 
                             metric='euclidean')
    X_tsne = tsne.fit_transform(X_reduced)

    label_encoder = LabelEncoder()
    encoded_true = label_encoder.fit_transform(true_labels)
    unique_true_labels = label_encoder.classes_
    n_true_classes = len(unique_true_labels)
    
    print(f"Количество семантических классов: {n_true_classes}")

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    unique_clusters = np.unique(cluster_labels)
    n_clusters = len(unique_clusters) - (1 if -1 in unique_clusters else 0)
    colors = generate_distinct_colors(n_clusters + 1)
    
    # if -1 in cluster_labels:
    #     noise_mask = cluster_labels == -1
    #     cluster_mask = ~noise_mask
    #     scatter1 = axes[0].scatter(X_tsne[cluster_mask, 0], X_tsne[cluster_mask, 1],
    #                                c=cluster_labels[cluster_mask], cmap=colors,
    #                                alpha=0.7, s=5, edgecolors='w', linewidths=0.1)
    #     axes[0].scatter(X_tsne[noise_mask, 0], X_tsne[noise_mask, 1],
    #                     c='gray', alpha=0.5, s=3, label=f'Шум ({sum(noise_mask)})')
    #     axes[0].legend()
    # else:
    #     scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels,
    #                                cmap='tab20', alpha=0.7, s=5, 
    #                                edgecolors='w', linewidths=0.1)

    for i, class_idx in enumerate(unique_clusters):
        mask = cluster_labels == class_idx
        if np.any(mask):
            axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                           c=[colors[i]], 
                            # alpha=0.7, 
                            s=5,
                           edgecolors='w', linewidths=0.1,
                           label=f'{class_idx}')
    # axes[0].legend()
    axes[0].set_title(f'Кластеры\n{n_clusters} кластеров', fontsize=14)
    axes[0].set_xlabel('t-SNE 1')
    axes[0].set_ylabel('t-SNE 2')
    # plt.colorbar(scatter1, ax=axes[0], label='Кластер')
    
    colors = generate_distinct_colors(n_true_classes)
    for i, class_idx in enumerate(unique_true_labels):
        mask = true_labels == class_idx
        if np.any(mask):
            axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                           c=[colors[i]], 
                            # alpha=0.7,
                            s=5,
                           edgecolors='w', linewidths=0.1,
                           label=f'{class_idx}')
    
    axes[1].set_title(f'Семантические классы\n{n_true_classes} классов', fontsize=14)
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    # axes[1].legend()
    plt.suptitle(f'Визуализация {len(X)} токенов', fontsize=16)
    # plt.legend()
    plt.tight_layout()
    
    if save_path:
        filepath = os.path.join(save_path, 'clusters_2d_visualization.png')
        plt.savefig(filepath, dpi=150, bbox_inches='tight')
        print(f"Визуализация сохранена: {filepath}")
    
    plt.show()
    
    return X_tsne

In [17]:
def visualize_full_tsne(X, y_upos, y_sc, output_dir):   
    ensure_directory_exists(output_dir)
    
    print(f"Всего токенов: {len(X)}")
    print(f"Уникальных UPOS: {np.unique(y_upos)}")
    # print(f"Уникальных SC: {np.unique(y_sc)}")
    
    unique_upos = np.unique(y_upos)
    unique_sc = np.unique(y_sc)    
    perplexity = 50
    print(f"Выполнение t-SNE для всех данных (perplexity={perplexity})...")
    
    tsne = TSNE(n_components=2, 
                random_state=100, 
                perplexity=perplexity,
                max_iter=1000, 
                learning_rate='auto', 
                init='random', 
                metric='euclidean')
    
    X_tsne_all = tsne.fit_transform(X)
    fig, ax = plt.subplots(1, 1, figsize=(16, 12))
    colors = generate_distinct_colors(len(unique_upos))
    color_map = {upos: colors[i] for i, upos in enumerate(unique_upos)}
    
    for i, upos in enumerate(unique_upos):
        mask = y_upos == upos
        if np.any(mask):
            ax.scatter(X_tsne_all[mask, 0], X_tsne_all[mask, 1],
                      c=[color_map[upos]], s=3, alpha=0.6,
                      edgecolors='none', label=f'{upos} ({np.sum(mask)})')
    
    ax.set_title(f'Все UPOS классы\nВсего токенов: {len(X)}, UPOS классов: {len(unique_upos)}', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('t-SNE 1', fontsize=12)
    ax.set_ylabel('t-SNE 2', fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_path_all = os.path.join(output_dir, 'all_upos_classes.png')
    plt.savefig(save_path_all, dpi=150, bbox_inches='tight')
    print(f"Общая визуализация сохранена: {save_path_all}")
    plt.show()

    fig, ax = plt.subplots(1, 1, figsize=(16, 12))
    colors = generate_distinct_colors(len(unique_sc))
    color_map = {sc: colors[i] for i, sc in enumerate(unique_sc)}
    for i, sc in enumerate(unique_sc):
        mask = y_sc == sc
        if np.any(mask):
            ax.scatter(X_tsne_all[mask, 0], X_tsne_all[mask, 1],
                      c=[color_map[sc]], s=3, alpha=0.6,
                      edgecolors='none')
    
    ax.set_title(f'Все Семантические классы\nВсего токенов: {len(X)}, сем классов: {len(unique_sc)}', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('t-SNE 1', fontsize=12)
    ax.set_ylabel('t-SNE 2', fontsize=12)
    # ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    save_path_all = os.path.join(output_dir, 'all_semantic_classes.png')
    plt.savefig(save_path_all, dpi=150, bbox_inches='tight')
    print(f"Общая визуализация сохранена: {save_path_all}")
    plt.show()

## running utils

In [18]:
def run_clustering_pipeline(embeddings: Dict,
                           method: str = 'hdbscan',
                           clustering_params: Optional[Dict] = None,
                           min_freq_threshold: int = 3,
                           normalization: Optional[str] = 'standard',
                           dimension_reduction: Optional[str] = 'pca',
                           n_components: int = 50,
                           output_dir: str = './clustering_results',
                           experiment_name: Optional[str] = None,
                           save_results_flag: bool = True,
                           visualize: bool = True,
                           is_ortho=False) -> Dict:

    ensure_directory_exists(output_dir)
    if experiment_name is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        experiment_name = f"{method}_{min_freq_threshold}_{timestamp}"
    experiment_dir = os.path.join(output_dir, experiment_name)
    ensure_directory_exists(experiment_dir)
    
    print("ЗАПУСК ПАЙПЛАЙНА КЛАСТЕРИЗАЦИИ")
    print(f"Метод: {method}")
    print(f"Порог частоты: >= {min_freq_threshold}")
    print(f"Директория эксперимента: {experiment_dir}")
    
    # 1. Подготовка и фильтрация данных
    X_raw, true_labels, tokens_info, indices, class_counts = filter_tokens_by_frequency(
        embeddings, min_freq_threshold
    )
    
    if len(X_raw) == 0:
        print("ОШИБКА: Нет данных для кластеризации!")
        return {}
    
    # 2. Нормализация
    X_norm = normalize_embeddings(X_raw, method=normalization)
    
    # 3. Снижение размерности
    X, reduction_info = reduce_dimensions(X_norm, method=dimension_reduction, n_components=n_components)
    if is_ortho:
        X = orthogonalize_by_pos(X_norm, true_labels)
    
    # 4. Подготовка параметров кластеризации
    if clustering_params is None:
        # Параметры по умолчанию для каждого метода
        if method == 'kmeans':
            n_clusters_est = min(50, len(np.unique(true_labels)))
            clustering_params = ClusteringFactory.create_kmeans_params(n_clusters=n_clusters_est)
        elif method == 'dbscan':
            clustering_params = ClusteringFactory.create_dbscan_params(eps=0.5, min_samples=5)
        elif method == 'hdbscan':
            clustering_params = ClusteringFactory.create_hdbscan_params(min_cluster_size=15)
        elif method == 'agglomerative':
            n_clusters_est = min(50, len(np.unique(true_labels)))
            clustering_params = ClusteringFactory.create_agglomerative_params(n_clusters=n_clusters_est)
    
    # 5. Кластеризация
    cluster_labels, clusterer, metrics = perform_clustering(
        X, method=method, params=clustering_params, true_labels=true_labels
    )
    
    if cluster_labels is None:
        print("ОШИБКА: Кластеризация не удалась!")
        return {}
    
    # 6. Анализ кластеров
    df_clusters, cluster_details = analyze_clusters_content(
        tokens_info, cluster_labels, true_labels
    )
    
    # 7. Визуализация 
    X_tsne = None
    if visualize:
        X_tsne = visualize_clusters_2d(X, cluster_labels, true_labels, save_path=experiment_dir)
        
        if len(np.unique(true_labels)) >= 5:
            visualize_confusion_matrix(cluster_labels, true_labels, save_path=experiment_dir)
    
    # 8. Сбор всех параметров для логирования
    all_params = {
        'method': method,
        'clustering_params': clustering_params,
        'min_freq_threshold': min_freq_threshold,
        'normalization': normalization,
        'dimension_reduction': dimension_reduction,
        'n_components': n_components,
        'experiment_name': experiment_name,
        'output_dir': output_dir,
        'save_results_flag': save_results_flag,
        'visualize': visualize
    }
    
    # 9. Сбор результатов
    results = {
        'X_raw': X_raw,
        'X': X,
        'true_labels': true_labels,
        'cluster_labels': cluster_labels,
        'tokens_info': tokens_info,
        'clusterer': clusterer,
        'metrics': metrics,
        'df_clusters': df_clusters,
        'cluster_details': cluster_details,
        'X_tsne': X_tsne,
        'class_counts': class_counts,
        'params': all_params,
        'reduction_info': reduction_info
    }
    
    # 10. Сохранение результатов
    if save_results_flag:
        save_dir = save_results(results, experiment_dir, experiment_name)
        results['save_dir'] = save_dir
        
        # Дополнительно сохраняем все параметры в отдельный файл для удобства
        params_file = os.path.join(experiment_dir, 'experiment_params.json')
        with open(params_file, 'w', encoding='utf-8') as f:
            # Конвертируем numpy типы в сериализуемые
            serializable_params = {}
            for k, v in all_params.items():
                if isinstance(v, np.integer):
                    serializable_params[k] = int(v)
                elif isinstance(v, np.floating):
                    serializable_params[k] = float(v)
                elif isinstance(v, dict):
                    serializable_params[k] = {k2: (int(v2) if isinstance(v2, np.integer) else 
                                                   float(v2) if isinstance(v2, np.floating) else v2)
                                             for k2, v2 in v.items()}
                else:
                    serializable_params[k] = v
            json.dump(serializable_params, f, indent=2, ensure_ascii=False)
        print(f"Параметры эксперимента сохранены: {params_file}")
    
    # 11. Сводка
    print("\n" + "=" * 80)
    print("СВОДКА РЕЗУЛЬТАТОВ")
    print(f"Эксперимент: {experiment_name}")
    print(f"Метод: {method}")
    print(f"Параметры кластеризации: {clustering_params}")
    print(f"Нормализация: {normalization}")
    print(f"Снижение размерности: {dimension_reduction} -> {n_components} компонент")
    print(f"Токенов: {len(X)}")
    print(f"Семантических классов: {len(np.unique(true_labels))}")
    print(f"Найдено кластеров: {metrics.get('n_clusters', 0)}")
    print(f"Шумовых точек: {metrics.get('n_noise', 0)} ({metrics.get('noise_percent', 0):.1f}%)")
    print(f"ARI: {metrics.get('ari', 0):.4f}")
    print(f"V-measure: {metrics.get('v_measure', 0):.4f}")
    if save_results_flag:
        print(f"Результаты сохранены в: {experiment_dir}")
    
    return results, clusterer

# START

In [19]:
import pickle

with open('overqwen_embeddings.pkl', 'rb') as f:
    all_embeddings = pickle.load(f)

In [ ]:
embeddings_upos = {}

null_count = 0
total_count = len(all_embeddings)

all_sc = {}

for key, data in all_embeddings.items():
    if isinstance(data, dict):
        form = data['form']
        upos = data['upos']
        sc = data['sem_class']

        if form == '#NULL' or data['sem_class'] == '_':
            null_count += 1
            continue  # Пропускаем #NULL токены
        
        if sc not in all_sc:
            all_sc[sc] = 0
        all_sc[sc] += 1
        
        # Добавляем в очищенный словарь
        embeddings_upos[key] = data
    else:
        print(f"Внимание: неправильный формат данных для ключа {key}")

print(f"Статистика очистки:")
print(f"  Исходное количество токенов: {total_count}")
print(f"  Удалено #NULL токенов: {null_count}")
print(f"  Оставлено токенов: {len(embeddings_upos)}")
print(f"  Процент удаленных: {null_count/total_count*100:.1f}%")

# all_embeddings = {}

In [ ]:
method = 'hdbscan'
clustering_params = {
    'min_cluster_size': 100, 
    'min_samples': 5, 
    'metric': 'euclidean', 
    'copy': False, 
    'core_dist_n_jobs': -1, 
    'approx_min_span_tree':True,
    'gen_min_span_tree': True,
    'cluster_selection_epsilon': 0.5, 
    'cluster_selection_method': 'leaf', 
    'alpha': 0.9,          
}

# method = 'kmeans'
# clustering_params = {'n_clusters': 600}

normalization_schema = 'l2'
dimension_reduction_method = 'pca'
n_comp = 50
min_freq_threshold = 10

embedding_type = 'qwen35_4b_middle'
output_dir = f'./{method}_{embedding_type}/'
experiment_name = 'after_transform'

results, clusterer = run_clustering_pipeline(
    embeddings=embeddings_upos,
    method=method,
    clustering_params=clustering_params,
    min_freq_threshold=min_freq_threshold,
    normalization=normalization_schema,
    dimension_reduction=dimension_reduction_method,
    n_components=n_comp,
    output_dir=output_dir,
    experiment_name=experiment_name,
    save_results_flag=True,
    visualize=True
)

In [ ]:
condensed_tree = clusterer.condensed_tree_

In [ ]:
condensed_tree.plot()

In [ ]:
clusterer.condensed_tree_.plot(select_clusters=True)

In [ ]:
# hierarchy_df = condensed_tree.to_pandas()
# hierarchy_df.to_csv('way.csv')

In [ ]:
visualize_full_tsne(results['X'], results['cluster_labels'], results['true_labels'], '/hdbscan_qwen35_4b_middle/final/visual')